In [0]:
from pyspark.sql.functions import col,when, round

# Lire la table bronze

In [0]:
df_bronze = spark.table("project_cardio.bronze.cardio_csv")

# Transformation et mettoyage des donnnees
## suppression des doublons

In [0]:
df_silver = (df_bronze.dropDuplicates(["id"]))

##filtrer les valeurs aberrantes 

In [0]:
df_silver = (df_silver.filter(col("height") > 0)
    .filter(col("weight") > 0)
    .filter(col("ap_hi") > 0)
    .filter(col("ap_lo") > 0)
    .filter(col("ap_hi") > col("ap_lo")))


##renommer les colonnes

In [0]:
df_silver = (df_silver.withColumnRenamed("ap_hi", "tension_systolic")
    .withColumnRenamed("ap_lo", "tension_diastolic"))

##convertir age en années

In [0]:
df_silver = df_silver.withColumn("age_years", round(col("age") / 365.25, 0).cast("int"))

##créer IMC

In [0]:
df_silver = df_silver.withColumn("bmi", round(col("weight") / ((col("height") / 100) ** 2), 2))

In [0]:
display(df_silver)

In [0]:
df_silver = df_silver.withColumn(
    "gender",
    when(col("gender") == 1, "male")
    .when(col("gender") == 2, "female")
    .otherwise("unknown")
)

display(df_silver)

# writting data en silver

In [0]:
df_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("project_cardio.silver.cardio_clean")